In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, DoubleType
from pyspark.ml.feature import VectorAssembler
import xgboost as xgb
import pandas as pd
import numpy as np

# -------------------------
# 1️⃣ Créer la session Spark
# -------------------------
spark = SparkSession.builder \
    .appName("KafkaSparkXGB") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# -------------------------
# 2️⃣ Définir le schéma JSON
# -------------------------
schema = StructType(
    [StructField("Time", DoubleType(), True)] +
    [StructField(f"V{i}", DoubleType(), True) for i in range(1, 29)] +
    [StructField("Amount", DoubleType(), True)]
)

# -------------------------
# 3️⃣ Lire le stream Kafka
# -------------------------
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "credit-card-transactions") \
    .option("startingOffsets", "earliest") \
    .load()

# -------------------------
# 4️⃣ Parser JSON et créer colonnes
# -------------------------
parsed_df = kafka_df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

# -------------------------
# 5️⃣ Feature Engineering
# -------------------------
feature_cols = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
features_df = assembler.transform(parsed_df)

# -------------------------
# 6️⃣ Charger le modèle XGBoost
# -------------------------
import joblib
xgb_model = joblib.load("../../models/xgboost_fraud_model2.pkl")

# -------------------------
# 7️⃣ Fonction de prédiction
# -------------------------
def predict_batch(batch_df, batch_id):
    if batch_df.rdd.isEmpty():
        return

    small_df = batch_df.select("features").limit(20)
    pdf = small_df.toPandas()

    X = np.array(pdf["features"].tolist()) # X have  one clomuns features that has all infos (data= time + V1 to V28 + Amount)
    probas = xgb_model.predict_proba(X)[:, 1]  # probabilité classe 1 (fraude)
    pdf["fraud_proba"] = probas
    pdf["prediction"] = (probas > 0.003).astype(int)  # seuil abaissé volontairement

    print(f"\n========== BATCH {batch_id} ==========")
    print(f"Rows received: {batch_df.count()}")
    print(pdf)



# -------------------------
# 8️⃣ Lancer le streaming avec prédiction
# -------------------------
query = features_df.writeStream \
    .foreachBatch(predict_batch) \
    .trigger(processingTime="1 second") \
    .start()


query.awaitTermination()


ERROR:root:KeyboardInterrupt while sending command.                             
Traceback (most recent call last):
  File "/home/souhail/.local/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/souhail/.local/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 